## Example 1: NaN-aware reductions give NumPy's answers


In [ ]:
import numpy as np
import numbagg

data = np.array([1.0, 2.5, np.nan, 4.2, 5.1, np.nan, 7.3, 8.9])
checks = {
    "nansum": (numbagg.nansum(data), np.nansum(data)),
    "nanmean": (numbagg.nanmean(data), np.nanmean(data)),
    "nanstd": (numbagg.nanstd(data), np.nanstd(data, ddof=1)),  # numbagg's default ddof is 1, NumPy's is 0
    "nanvar": (numbagg.nanvar(data, ddof=0), np.nanvar(data)),
    "nanmin": (numbagg.nanmin(data), np.nanmin(data)),
    "nanmax": (numbagg.nanmax(data), np.nanmax(data)),
    "nanmedian": (numbagg.nanmedian(data), np.nanmedian(data)),
    "nanquantile": (numbagg.nanquantile(data, 0.25), np.nanquantile(data, 0.25)),
    "nancount": (numbagg.nancount(data), np.count_nonzero(~np.isnan(data))),
}
for name, (ours, numpy_answer) in checks.items():
    print(f"  {name}: {float(ours):.6g}")
    assert np.isclose(ours, numpy_answer), (name, ours, numpy_answer)
print(f"numbagg {numbagg.__version__}: Example 1 completed successfully")

## Example 2: moving windows, with and without a minimum count


In [ ]:
# Example 2: moving-window sums and means against windows cut out with NumPy. By default a window needs all
# `window` values (`min_count=window`), so a window holding the NaN is NaN; with min_count=1 it uses the rest.
import numpy as np
import numbagg
from numpy.lib.stride_tricks import sliding_window_view

data = np.arange(1.0, 11.0)
data[4] = np.nan
window = 3
expected = np.full(data.shape, np.nan)
expected[window - 1 :] = sliding_window_view(data, window).mean(axis=-1)
np.testing.assert_allclose(numbagg.move_mean(data, window=window), expected, equal_nan=True)

windows = [data[max(0, i - window + 1) : i + 1] for i in range(len(data))]
np.testing.assert_allclose(numbagg.move_mean(data, window=window, min_count=1), [np.nanmean(w) for w in windows])
np.testing.assert_allclose(numbagg.move_sum(data, window=window, min_count=1), [np.nansum(w) for w in windows])
std = numbagg.move_std(data, window=window, min_count=2)
expected_std = [np.nanstd(w, ddof=1) if np.count_nonzero(~np.isnan(w)) >= 2 else np.nan for w in windows]
np.testing.assert_allclose(std, expected_std, equal_nan=True)
print("move_mean:", numbagg.move_mean(data, window=window))
print("Example 2 completed successfully")


## Example 3: group-by aggregations against np.bincount


In [ ]:
# Example 3: grouped aggregations. A label of -1 belongs to no group and is left out; NaN values are skipped.
import numpy as np
import numbagg

labels = np.array([0, 0, 1, 1, 2, 2, 0, 1, -1])
values = np.array([1.0, 2.0, 3.0, np.nan, 5.0, 6.0, 7.0, 8.0, 100.0])
kept = (labels >= 0) & ~np.isnan(values)
sums = np.bincount(labels[kept], weights=values[kept], minlength=3)
counts = np.bincount(labels[kept], minlength=3)

np.testing.assert_allclose(numbagg.group_nansum(values, labels), sums)
np.testing.assert_allclose(numbagg.group_nancount(values, labels), counts)
np.testing.assert_allclose(numbagg.group_nanmean(values, labels), sums / counts)
np.testing.assert_allclose(numbagg.group_nanmax(values, labels), [7.0, 8.0, 6.0])
np.testing.assert_allclose(numbagg.group_nanmin(values, labels), [1.0, 3.0, 5.0])
# num_labels widens the result: a group with no values gets 0 for a sum and NaN for a mean.
wide = numbagg.group_nanmean(values, labels, num_labels=4)
assert wide.shape == (4,) and np.isnan(wide[3]), wide
print("group_nanmean:", numbagg.group_nanmean(values, labels))
print("Example 3 completed successfully")


## Example 4: exponentially weighted moving mean


In [ ]:
# Example 4: an exponentially weighted moving mean, checked against the weighted average written out by hand
# (weight (1 - alpha) ** k for the value k steps back, the same as pandas' `ewm(alpha=..., adjust=True)`).
import numpy as np
import numbagg

rng = np.random.default_rng(0)
data = rng.normal(size=12)
alpha = 0.3
expected = []
for t in range(len(data)):
    weights = (1 - alpha) ** np.arange(t, -1, -1)
    expected.append(np.sum(weights * data[: t + 1]) / np.sum(weights))
np.testing.assert_allclose(numbagg.move_exp_nanmean(data, alpha=alpha), expected)

# Along an axis of a 2-D array, row by row.
grid = rng.normal(size=(3, 12))
by_row = numbagg.move_exp_nanmean(grid, alpha=alpha, axis=-1)
for row in range(3):
    np.testing.assert_allclose(by_row[row], numbagg.move_exp_nanmean(grid[row], alpha=alpha))
print("move_exp_nanmean:", np.round(numbagg.move_exp_nanmean(data, alpha=alpha)[:4], 4))
print("Example 4 completed successfully")


## Example 5: fill gaps, find extremes, and reduce many axes at once


In [ ]:
# Example 5: ffill/bfill with a limit, nanargmax/nanargmin, and a reduction over a 3-D array that numbagg runs
# in parallel across the kept axis; each answer is checked against NumPy.
import numpy as np
import numbagg

gaps = np.array([1.0, np.nan, np.nan, np.nan, 5.0, np.nan])
np.testing.assert_array_equal(numbagg.ffill(gaps), [1.0, 1.0, 1.0, 1.0, 5.0, 5.0])
np.testing.assert_array_equal(numbagg.ffill(gaps, limit=1), [1.0, 1.0, np.nan, np.nan, 5.0, 5.0])
np.testing.assert_array_equal(numbagg.bfill(gaps, limit=2), [1.0, np.nan, 5.0, 5.0, 5.0, np.nan])

rng = np.random.default_rng(1)
cube = rng.normal(size=(4, 50, 60))
cube[0, 3, 7] = np.nan
assert numbagg.nanargmax(cube[0, 3]) == np.nanargmax(cube[0, 3])
assert numbagg.nanargmin(cube[0, 3]) == np.nanargmin(cube[0, 3])
np.testing.assert_allclose(numbagg.nanmean(cube, axis=(1, 2)), np.nanmean(cube, axis=(1, 2)))
np.testing.assert_allclose(numbagg.nansum(cube, axis=0), np.nansum(cube, axis=0))
print("nanmean over axes 1 and 2:", np.round(numbagg.nanmean(cube, axis=(1, 2)), 4))
print("Example 5 completed successfully")
